# Notebook 15 – Model Failure Analysis

## Topic 0: Setup and the Failure Report Helper

**Concept**
Failure analysis means deliberately feeding a model inputs it is likely to get wrong, then recording the 6 facts of each failure: Input, Expected output, Actual output, Possible reason, Analysis, Possible improvement.

**Why it's needed**
A pre-trained model is **not automatically correct for every input**. Engineers who only look at success cases ship models that break in production. We build one helper, `report()`, so every failure in this notebook is documented in the same format.

In [ ]:
# Run once if needed:
# !pip install -q transformers torch datasets pandas

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from transformers import pipeline

FAILURES = []   # every analysed case is stored here for the final summary

def report(case_type, input_text, expected, actual, reason, analysis, improvement, failed=True):
    """Print one case in the 6-part failure format and store it."""
    status = "FAILURE" if failed else "model was correct this time"
    print("=" * 72)
    print(f"[{case_type}]  ->  {status}")
    print(f"1. Input               : {input_text}")
    print(f"2. Expected output     : {expected}")
    print(f"3. Actual output       : {actual}")
    print(f"4. Possible reason     : {reason}")
    print(f"5. Analysis            : {analysis}")
    print(f"6. Possible improvement: {improvement}")
    FAILURES.append({
        "Case": case_type, "Input": input_text, "Expected": expected,
        "Actual": actual, "Failed": failed,
    })

print("Setup done. Helper ready: report(...)")

### What this example tells us
Every failure will be printed in the same 6-part layout, and all cases are collected in `FAILURES` for a final summary table.

### AI/ML Example
In model evaluation, an error log with input, expected, actual and reason is the starting point for deciding whether to fix data, prompts, thresholds or the model.

### Business Example
A support team keeps a "wrong answers" sheet for its chatbot. Reviewing it weekly shows which customer questions need human hand-off.

### AI/ML Engineer Use Case
Build a reusable error-analysis template once, then use it for every model you evaluate before release.

## Topic 1: Incorrect Prediction (Sarcasm)

**Concept**
The model returns a clear label, but the label is wrong because the sentence means the opposite of its words (sarcasm, irony).

**Why it's needed**
Wrong predictions that look confident are the most dangerous failures, because nobody notices them. Sentiment models read surface words like "great" and "impressive" and can miss the real meaning.

In [ ]:
clf = pipeline("sentiment-analysis",
               model="distilbert-base-uncased-finetuned-sst-2-english")

cases = [
    ("Oh great, another delay. Just what I needed today.",
     "NEGATIVE", "Sarcasm: positive words ('great') with negative intent."),
    ("Wow, the support team took only three weeks to reply. Impressive!",
     "NEGATIVE", "Irony: 'only three weeks' and 'Impressive' are used sarcastically."),
    ("Thanks a lot for breaking my phone on the first day.",
     "NEGATIVE", "Polite words ('Thanks a lot') hide a complaint."),
]

for text, expected, reason in cases:
    result = clf(text)[0]
    actual = f"{result['label']} ({result['score']:.1%})"
    report(
        "Incorrect prediction", text, expected, actual, reason,
        "The model was trained on movie-review style text and mostly learned word-level sentiment cues, not tone.",
        "Fine-tune on sarcastic examples from your own domain, or add a sarcasm-detection step before sentiment.",
        failed=(result["label"] != expected),
    )

### What this example tells us
The model can give a confident label that is the opposite of the human meaning. A high score does not prove the answer is right.

### AI/ML Example
Sentiment classifiers on social media misread sarcastic tweets, which lowers accuracy on brand-monitoring data.

### Business Example
A company auto-tags angry customer emails as "happy" because they contain words like "thanks" and "great", so urgent complaints are not escalated.

### AI/ML Engineer Use Case
Build a small "hard cases" test set (sarcasm, negation) and run it before every model release.

## Topic 2: Unexpected Output (Text Generation)

**Concept**
The model produces text that is fluent but does not answer the task: wrong fact, repeated words, or an unrelated continuation.

**Why it's needed**
A base language model only predicts the next likely token. It was not trained to follow instructions or to know facts reliably, so its output can be completely unexpected.

In [ ]:
gen = pipeline("text-generation", model="gpt2")

cases = [
    ("The capital of Australia is", "canberra", "Canberra"),
    ("Translate to French: Good morning ->", "bonjour", "Bonjour"),
    ("Q: What is 17 + 25?\nA:", "42", "42"),
]

for prompt, keyword, expected in cases:
    out = gen(prompt, max_new_tokens=15, do_sample=False,
              pad_token_id=gen.tokenizer.eos_token_id)[0]["generated_text"]
    completion = out[len(prompt):].strip().replace("\n", " ")
    report(
        "Unexpected output", prompt.replace("\n", " "), expected, completion,
        "GPT-2 is a small base model: it continues text, it does not follow instructions or calculate.",
        "The output may be fluent, but fluency is not correctness. The model has no fact lookup or arithmetic ability.",
        "Use an instruction-tuned model, give examples in the prompt, or connect a calculator / knowledge source.",
        failed=(keyword not in completion.lower()),
    )

### What this example tells us
Fluent text can still be wrong. The model completes patterns; it does not verify facts.

### AI/ML Example
Hallucination in language models: a confident but false answer to a factual question.

### Business Example
A chatbot built on a base model invents a refund policy that the company does not have, and customers quote it back.

### AI/ML Engineer Use Case
Choose instruction-tuned models for Q&A, and add retrieval or tools when answers must be factually correct.

## Topic 3: Low-Confidence Output

**Concept**
The model gives a label, but the probability behind it is low, meaning the model is unsure. Mixed or neutral text often causes this.

**Why it's needed**
A prediction without its confidence is half the story. Checking the score lets us send uncertain cases to a human instead of trusting them blindly.

In [ ]:
THRESHOLD = 0.90   # below this, we treat the answer as "not trustworthy"

texts = [
    "The food was great but the service was terrible.",
    "It was okay, I guess.",
    "The product arrived on Tuesday.",
]

for text in texts:
    result = clf(text)[0]
    low = result["score"] < THRESHOLD
    report(
        "Low-confidence output", text,
        "NEUTRAL / MIXED (model only has POSITIVE and NEGATIVE)",
        f"{result['label']} ({result['score']:.1%})",
        "The model has only 2 classes, so mixed or neutral text is forced into one of them.",
        f"Threshold = {THRESHOLD:.0%}. Score below it means the model is unsure; a high score on neutral text would still be a forced choice.",
        "Add a confidence threshold with human review, or use a model with a NEUTRAL class.",
        failed=low,
    )

### What this example tells us
A score near 50-70% says "unsure". Even a high score can hide a forced choice when the right answer is not among the labels.

### AI/ML Example
Production classifiers use confidence thresholds and send low-score items to an "uncertain" queue.

### Business Example
An email triage system auto-routes only messages above 90% confidence; the rest go to a human agent.

### AI/ML Engineer Use Case
Pick the threshold from validation data, balancing accuracy against how many cases need manual review.

## Topic 4: Ambiguous Output (Same Word, Different Meaning)

**Concept**
One word can mean different things (Jordan = person or country, Amazon = river or company). The model must use context to decide.

**Why it's needed**
Ambiguity is a limit of language, not a bug. We need to know when a model cannot separate meanings so we do not trust a single label per word.

In [ ]:
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

cases = [
    ("Jordan visited Jordan last year.", "jordan", ["PER", "LOC"]),
    ("Amazon is a river, and Amazon is also a company.", "amazon", ["LOC", "ORG"]),
]

for text, target, expected_labels in cases:
    entities = ner(text)
    found = [(e["entity_group"], round(float(e["score"]), 2))
             for e in entities if e["word"].strip().lower() == target]
    labels = [label for label, _ in found]
    report(
        "Ambiguous output", text,
        f"'{target}' -> {expected_labels} (in order)",
        f"'{target}' -> {found}",
        "The same word appears twice with different meanings; the model may give both the same label or a weak score.",
        "Context decides the meaning. Short sentences give very little context, so predictions for repeated words are unreliable.",
        "Provide more context, use entity linking to a knowledge base, or flag repeated surface words for review.",
        failed=(labels != expected_labels),
    )

### What this example tells us
When the same word has several meanings, a token-level model may treat them identically. Check the label AND the score.

### AI/ML Example
Entity linking systems connect "Jordan" to the right person or country using surrounding context.

### Business Example
A news-monitoring tool counts "Amazon" mentions for the company brand but also picks up articles about the rainforest.

### AI/ML Engineer Use Case
Log ambiguous words found in production and add context or a disambiguation step for them.

## Topic 5: Poor Transcription (Noisy Audio)

**Concept**
A speech-to-text model converts audio into text. When the audio is noisy, the transcript gets words wrong or missing. We measure this with **WER** (Word Error Rate): lower is better, 0.0 is perfect.

**Why it's needed**
Models are tested on clean audio but used in noisy places (calls, streets, offices). We must measure how fast quality drops.

In [ ]:
from datasets import load_dataset

asr = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en")

def normalize(text):
    return re.sub(r"[^a-z' ]", "", text.lower()).strip()

def word_error_rate(ref, hyp):
    r, h = normalize(ref).split(), normalize(hyp).split()
    d = np.zeros((len(r) + 1, len(h) + 1), dtype=int)
    d[:, 0] = range(len(r) + 1)
    d[0, :] = range(len(h) + 1)
    for i in range(1, len(r) + 1):
        for j in range(1, len(h) + 1):
            cost = 0 if r[i - 1] == h[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + cost)
    return d[len(r), len(h)] / max(len(r), 1)

def add_noise(signal, snr_db):
    rng = np.random.default_rng(0)
    noise = rng.normal(0, 1, len(signal))
    target = np.mean(signal ** 2) / (10 ** (snr_db / 10))
    return (signal + noise * np.sqrt(target)).astype("float32")

# One clean speech sample with a known reference transcript
ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = ds[0]
audio = sample["audio"]
if isinstance(audio, dict):
    signal, sr = np.asarray(audio["array"], dtype="float32"), audio["sampling_rate"]
else:  # newer `datasets` versions return a decoder object
    s = audio.get_all_samples()
    signal, sr = s.data.squeeze().numpy().astype("float32"), s.sample_rate
reference = sample["text"]

clean_text = asr({"raw": signal, "sampling_rate": sr})["text"]
noisy_text = asr({"raw": add_noise(signal, snr_db=-5), "sampling_rate": sr})["text"]

clean_wer = word_error_rate(reference, clean_text)
noisy_wer = word_error_rate(reference, noisy_text)
print(f"WER on clean audio : {clean_wer:.2f}")
print(f"WER on noisy audio : {noisy_wer:.2f}")

report(
    "Poor transcription", "Speech audio with added noise (SNR = -5 dB)",
    reference.lower(), noisy_text.strip(),
    "Noise hides parts of the speech, and the model was mostly trained on cleaner audio.",
    f"WER moved from {clean_wer:.2f} (clean) to {noisy_wer:.2f} (noisy). Higher WER means more wrong, missing or extra words.",
    "Apply noise reduction, fine-tune on noisy audio, use a larger model, or ask the speaker to repeat when quality is low.",
    failed=(noisy_wer > 0.20),
)

### What this example tells us
The same model gives a good transcript on clean audio and a worse one on noisy audio. Test quality under realistic conditions.

### AI/ML Example
Speech models are evaluated with WER on clean and noisy test sets, and also across accents and microphones.

### Business Example
A call-centre tool transcribes calls. Calls from mobile phones in traffic produce poor text, so the compliance summaries miss key words.

### AI/ML Engineer Use Case
Track WER per audio condition (noise level, accent, microphone) and set a minimum quality gate before using transcripts downstream.

## Topic 6: Incorrect Entity (Lowercase and Misspelled Text)

**Concept**
A named-entity model marks names of people, organisations and locations. It fails when the entity is missed, given the wrong type, or has the wrong boundary.

**Why it's needed**
NER models often depend on capital letters and correct spelling. Real user text is often lowercase, misspelled or informal, so entities get lost.

In [ ]:
cases = [
    ("sundar pichai works at google in mountain view",
     {"sundar pichai": "PER", "google": "ORG", "mountain view": "LOC"},
     "Lowercase text: the cased model relies on capital letters as a strong clue."),
    ("Barak Obama visited Microsft in Seatle.",
     {"barak obama": "PER", "microsft": "ORG", "seatle": "LOC"},
     "Misspelled names are split into unfamiliar sub-word pieces."),
]

for text, expected, reason in cases:
    predicted = {e["word"].strip().lower(): e["entity_group"] for e in ner(text)}
    report(
        "Incorrect entity", text, expected, predicted, reason,
        "Compare the two dictionaries: missing keys mean missed entities, different values mean wrong types, odd keys mean wrong boundaries.",
        "Normalise text (casing, spelling), fine-tune on informal text, or use a model trained on noisy data.",
        failed=(predicted != expected),
    )

### What this example tells us
Entity extraction quality depends on how clean the text is. The same sentence can be handled well or badly depending on casing and spelling.

### AI/ML Example
NER on chat messages and social media performs worse than on news text, because the model was trained mostly on formal writing.

### Business Example
A CRM tool extracts customer and company names from emails. Lowercase, hurried messages cause missed names and incomplete records.

### AI/ML Engineer Use Case
Test NER on your real, messy data, not only on benchmark text, and add a text-cleaning step if needed.

## Topic 7: Incorrect Classification (Domain Shift)

**Concept**
The model assigns the wrong label because the same word carries a different meaning in a different domain. In medicine, "negative" is good news; in reviews, it is bad.

**Why it's needed**
Models only know the domain they were trained on. Using a movie-review model on medical, legal or slang text gives wrong classes without any warning.

In [ ]:
cases = [
    ("The biopsy result came back negative, so the doctor said I am cancer-free.",
     "POSITIVE", "Medical domain: 'negative' and 'cancer' are good-news words here."),
    ("Great news, my COVID test was negative.",
     "POSITIVE", "The word 'negative' pulls the model towards NEGATIVE despite 'Great news'."),
    ("This game is sick! Absolutely wicked graphics.",
     "POSITIVE", "Slang: 'sick' and 'wicked' are praise for gamers."),
]

for text, expected, reason in cases:
    result = clf(text)[0]
    report(
        "Incorrect classification", text, expected,
        f"{result['label']} ({result['score']:.1%})", reason,
        "The model learned word meanings from movie reviews. Out-of-domain vocabulary breaks those learned associations.",
        "Use a domain-specific model, fine-tune on labelled in-domain data, or evaluate on a domain test set first.",
        failed=(result["label"] != expected),
    )

### What this example tells us
The label set is fine, but the model's word knowledge does not match the domain. Always validate on data that looks like production data.

### AI/ML Example
Domain adaptation: a general sentiment model is fine-tuned on clinical notes or gaming chat to learn domain meanings.

### Business Example
A hospital feedback tool marks "test came back negative" messages as unhappy patients and sends them to the complaints team.

### AI/ML Engineer Use Case
Before deploying a pre-trained model, run a domain check with 50-100 real examples and compare accuracy to the benchmark claim.

## Topic 8: Failure Summary and Key Lesson

**Concept**
Put every analysed case in one table and count how many were real failures.

**Why it's needed**
A summary table shows patterns across failure types, which tells us where to invest: more data, better prompts, thresholds or a different model.

In [ ]:
df = pd.DataFrame(FAILURES)
df["Input"] = df["Input"].astype(str).str.slice(0, 40)
df["Expected"] = df["Expected"].astype(str).str.slice(0, 30)
df["Actual"] = df["Actual"].astype(str).str.slice(0, 30)
print(df.to_string(index=False))

print("\nFailures per case type:")
print(df.groupby("Case")["Failed"].agg(failed="sum", total="count").to_string())

print("\nKEY LESSON: A pre-trained model is not automatically correct for every input.")
print("Always test on hard cases, check confidence, and measure on data like your real users' data.")

### What this example tells us
Failures cluster in predictable places: sarcasm, noise, domain words, ambiguity and unusual formatting. These can be tested for in advance.

### AI/ML Example
A model card lists known limitations and failure modes next to the accuracy numbers.

### Business Example
Before launch, a product team reviews the failure table and decides which cases need a human fallback.

### AI/ML Engineer Use Case
Make failure analysis a mandatory step in every project: collect failures, find the pattern, then fix data, thresholds, prompts or models, and re-test.